In [ ]:
from pathlib import Path
from collections import Counter
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
ROOT = Path('..').resolve()
FISHER_TAB = ROOT / 'results' / 'fisher_day0_day15' / 'tables'
OUT = ROOT / 'coursework_submission_notebooks' / 'outputs' / '06_motifs'
TAB = OUT / 'tables'
FIG = OUT / 'figures'
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
DONORS = ['P1', 'P2', 'Q1', 'Q2', 'S1', 'S2']
AA_RE = re.compile('^[ACDEFGHIKLMNPQRSTVWY*]+$')


In [ ]:
def gene_root(x):
    s = str(x)
    return s.split('*')[0] if s and s != 'nan' else 'NA'

def extract_kmers(seq, k):
    seq = str(seq)
    if not AA_RE.match(seq) or len(seq) < k:
        return []
    return [seq[i:i + k] for i in range(len(seq) - k + 1)]

def kmer_presence(seqs, k):
    c = Counter()
    n = 0
    for s in seqs:
        ks = extract_kmers(s, k)
        if not ks:
            continue
        n += 1
        for km in set(ks):
            c[km] += 1
    return (c, n)

def enrich_table(fg_counts, n_fg, bg_counts, n_bg, min_fg=5):
    rows = []
    for key in set(fg_counts) | set(bg_counts):
        a = fg_counts.get(key, 0)
        if a < min_fg:
            continue
        b = n_fg - a
        c = bg_counts.get(key, 0)
        d = n_bg - c
        (oddsr, p) = fisher_exact([[a, b], [c, d]], alternative='greater')
        rows.append({'feature': key, 'n_fg': a, 'n_fg_total': n_fg, 'frac_fg': a / n_fg if n_fg else np.nan, 'n_bg': c, 'n_bg_total': n_bg, 'frac_bg': c / n_bg if n_bg else np.nan, 'odds_ratio': oddsr, 'p_value': p})
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out
    (_, q, _, _) = multipletests(out['p_value'], method='fdr_bh')
    out['q_value'] = q
    out['enrichment'] = out['frac_fg'] / out['frac_bg'].replace(0, np.nan)
    return out.sort_values(['q_value', 'odds_ratio'], ascending=[True, False])


In [ ]:
v_fg = Counter()
v_bg = Counter()
j_fg = Counter()
j_bg = Counter()
n_fg = 0
n_bg = 0
fg_aa = []
bg_aa = []
for donor in DONORS:
    df = pd.read_parquet(FISHER_TAB / f'fisher_all_clonotypes_{donor}.parquet')
    df['V'] = df['V_gene'].map(gene_root)
    df['J'] = df['J_gene'].map(gene_root)
    mark = df['fisher_positive_FC4'].fillna(False).astype(bool)
    fg = df[mark]
    bg = df[~mark]
    n_fg += len(fg)
    n_bg += len(bg)
    v_fg.update(fg['V'])
    v_bg.update(bg['V'])
    j_fg.update(fg['J'])
    j_bg.update(bg['J'])
    fg_aa.extend(fg['cdr3_aa'].astype(str).tolist())
    bg_aa.extend(bg['cdr3_aa'].astype(str).tolist())
vj_v = enrich_table(v_fg, n_fg, v_bg, n_bg, min_fg=10)
vj_v['gene_type'] = 'V'
vj_j = enrich_table(j_fg, n_fg, j_bg, n_bg, min_fg=10)
vj_j['gene_type'] = 'J'
vj = pd.concat([vj_v, vj_j], ignore_index=True)
vj.to_csv(TAB / 'vj_enrichment.csv', index=False)
vj[vj['q_value'] < 0.05].head(20)


In [ ]:
(k2_fg, n2_fg) = kmer_presence(fg_aa, 2)
(k2_bg, n2_bg) = kmer_presence(bg_aa, 2)
(k3_fg, n3_fg) = kmer_presence(fg_aa, 3)
(k3_bg, n3_bg) = kmer_presence(bg_aa, 3)
kmer2 = enrich_table(k2_fg, n2_fg, k2_bg, n2_bg, min_fg=20)
kmer2['k'] = 2
kmer3 = enrich_table(k3_fg, n3_fg, k3_bg, n3_bg, min_fg=20)
kmer3['k'] = 3
kmers = pd.concat([kmer2, kmer3], ignore_index=True)
kmers.to_csv(TAB / 'kmer_enrichment.csv', index=False)
top_v = vj_v.nsmallest(15, 'q_value')
top_k = kmer2.nsmallest(15, 'q_value')
(fig, axes) = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh(top_v['feature'][::-1], top_v['odds_ratio'][::-1], color='#4c72b0')

axes[1].barh(top_k['feature'][::-1], top_k['odds_ratio'][::-1], color='#e07b39')

plt.tight_layout()
plt.savefig(FIG / 'vj_kmer_enrichment.png', dpi=150, bbox_inches='tight')
plt.show()
kmers[kmers['q_value'] < 0.05].head(15)
